In [4]:
!pip install --upgrade accelerate datasets transformers

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached tokenizers-0.22.1-cp39-abi3-macosx_11_0_arm64.whl (2.9 MB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.1.5
    Uninstalling huggingface_hub-1.1.5:
      Successfully uninstalled huggingface_hub-1.1.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]


In [ ]:
import re
import json
import torch
import openai
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from datasets import load_dataset
from huggingface_hub import login
from transformers import pipeline
from utils.system_instructions import SYSTEM_PROMPTS
from config import openai_token, hf_token, BASE_DIR, OUTPUTS_DIR, DATASETS_DIR

/Users/darpanaswal/Work/InterpretableSymGuard/symguard/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
HF_TOKEN = hf_token
OPENAI_API_KEY = openai_token
login(token=HF_TOKEN)

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

gpt_client = OpenAI(api_key=OPENAI_API_KEY)

In [16]:
def clean_category(row):
    category = row['category']
    if isinstance(category, dict):
        if all(value is False for value in category.values()):
            return "safe"
        for key, value in category.items():
            if value:
                return key
    return None

In [52]:
df_tech = pd.read_csv("hf://datasets/SoftMINER-Group/TechHazardQA/TechHAZARDQA.csv")
df_tech = df_tech[['Question', 'Topic']].rename(columns={'Question': 'text', 'Topic': 'label'})
df_tech = df_tech.sample(500, random_state=42)

df_adv_bench = pd.read_parquet("hf://datasets/walledai/AdvBench/data/train-00000-of-00001.parquet")
df_adv_bench = df_adv_bench[['prompt']].rename(columns={'prompt': 'text'})
df_adv_bench = df_adv_bench.sample(500, random_state=42)

splits = {'en': 'data/catqa_english.json', 'zh': 'data/catqa_chinese.json', 'vi': 'data/catqa_vietnamese.json'}
df_cat = pd.read_json("hf://datasets/declare-lab/CategoricalHarmfulQA/" + splits["en"], lines=True)
df_cat = df_cat[['Question', 'Subcategory']].rename(columns={'Question': 'text', 'Subcategory': 'label'})
df_cat = df_cat.sample(500, random_state=42)

In [ ]:
# label every prompt in df_adv_bench with using GPT
def label_adv_bench(df):
    labels = []
    for text in tqdm(df['text'], desc="Labeling AdvBench prompts"):
        response = gpt_client.chat.completions.create(
            model="gpt-5.1",
            messages=[
                {"role": "system", "content": """You are a jailbreak prompt classifier. Your task is to generate a label for a given harmful prompt. Taxonomy available: "profanity", "graphic depictions", "threatening behavior", "misinformation", "discrimination", "cybercrime", "dangerous or illegal suggestions". Output nothing but the label."""},
                {"role": "user", "content": f"Categorize the prompt into a high level jailbreak category: {text}"}
            ],
            max_completion_tokens=10,
            temperature=0
        )
        response = response.choices[0].message.content.strip()
        # Clean the response to ensure it matches the expected categories
        response = re.sub(r'\W+', ' ', response).strip().lower()
        labels.append(response)
    return labels

df_adv_bench['label'] = label_adv_bench(df_adv_bench)

# Harmful dataset

In [57]:
df_harmful_train = pd.concat([df_tech, df_adv_bench, df_cat]).reset_index(drop=True)

In [58]:
df_harm = pd.read_json("hf://datasets/declare-lab/HarmfulQA/data_for_hub.json")
df_harm = df_harm[['question']].rename(columns={'question': 'text'})
df_harmful_test = df_harm.sample(500, random_state=42)

In [59]:
df_harmful_train.to_csv(f"{BASE_DIR}/data/datasets/harmful_train.csv", index=False)
df_harmful_test.to_csv(f"{BASE_DIR}/data/datasets/harmful_test.csv", index=False)

# Safe

In [ ]:
df_alpaca = pd.read_json("hf://datasets/yahma/alpaca-cleaned/alpaca_data_cleaned.json")
df_safe = df_alpaca[['instruction']].rename(columns={'instruction': 'text'})
df_safe_train = df_safe.sample(1500, random_state=42)

In [ ]:
df_sae_train = pd.concat([df_harmful_train, df_safe_train]).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
def generate_safe_queries(gpt_client):
    system_prompt = SYSTEM_PROMPTS["generate_safe"]
    try:
        response = gpt_client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "system", "content": system_prompt}],
            response_format={"type": "json_object"},   # <-- forces strict JSON
            max_completion_tokens=100000,
        )
        
        json_text = response.choices[0].message.content
        return json.loads(json_text)

    except Exception as e:
        print("Error in GPT evaluation:", e)
        return {}
    
safe_queries = generate_safe_queries(gpt_client)

In [ ]:
df_safe_test = pd.DataFrame.from_dict(safe_queries, orient='index', columns=['text'])

In [ ]:
df_sae_train.to_csv(f"{DATASETS_DIR}/sae_train.csv", index=False)
df_harmful_train.to_csv(f"{DATASETS_DIR}/harmful_train.csv", index=False)
df_harmful_test.to_csv(f"{DATASETS_DIR}/harmful_test.csv", index=False)
df_safe_train.to_csv(f"{DATASETS_DIR}/safe_train.csv", index=False)
df_safe_test.to_csv(f"{DATASETS_DIR}/safe_test.csv", index=False)

# Categorize labels of harmful train set

In [ ]:
df_sae_train = pd.read_csv(f"{DATASETS_DIR}/sae_train.csv")

In [68]:
labels = df_sae_train['label'].unique().tolist()
labels

['safe',
 'Intellectual property theft',
 'Bio Technology, Biology, Genetic Engineering',
 'Theft and Robbery',
 'Tracking and location monitoring',
 'Public Healthcare System, Pharmacology',
 'Finance, Banking',
 'misinformation',
 'dangerous or illegal suggestions',
 'Religious intolerance',
 'Social Media',
 'Nuclear Technology, Bio Nuclear Terrorism, Terrorism',
 'Gender-based violence',
 'Racial or ethnic hate',
 'Chemical Weapon, Biological and Chemical Weapons',
 'Political activism',
 'Real estate investment',
 'cybercrime',
 'Lobbying',
 'Debt management',
 'Counterfeiting',
 'Emotional and psychological abuse',
 'Unsafe practices',
 'Election advertising',
 'Self-harm',
 'Cyber Security',
 'Political misinformation',
 'Scamming',
 'Partisan commentary',
 'Investment scams',
 'Neglect',
 'False advertising',
 'Retirement planning',
 'Market manipulation',
 'discrimination',
 'Terrorism and extremism',
 'Employment scams',
 'Human Trafficking',
 'Nude imagery',
 'Hazardous mate

In [69]:
labels_str = ', '.join([f'"{label}"' for label in labels])
print(labels_str)

"safe", "Intellectual property theft", "Bio Technology, Biology, Genetic Engineering", "Theft and Robbery", "Tracking and location monitoring", "Public Healthcare System, Pharmacology", "Finance, Banking", "misinformation", "dangerous or illegal suggestions", "Religious intolerance", "Social Media", "Nuclear Technology, Bio Nuclear Terrorism, Terrorism", "Gender-based violence", "Racial or ethnic hate", "Chemical Weapon, Biological and Chemical Weapons", "Political activism", "Real estate investment", "cybercrime", "Lobbying", "Debt management", "Counterfeiting", "Emotional and psychological abuse", "Unsafe practices", "Election advertising", "Self-harm", "Cyber Security", "Political misinformation", "Scamming", "Partisan commentary", "Investment scams", "Neglect", "False advertising", "Retirement planning", "Market manipulation", "discrimination", "Terrorism and extremism", "Employment scams", "Human Trafficking", "Nude imagery", "Hazardous material handling", "Assault and battery", "

In [ ]:
def generate_broad_categories(labels_str, gpt_client):
    system_prompt = (SYSTEM_PROMPTS["generate_broad_cat"] + labels_str)
    try:
        response = gpt_client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "system", "content": system_prompt}],
            response_format={"type": "json_object"},   # <-- forces strict JSON
            max_completion_tokens=10000,
        )
        
        json_text = response.choices[0].message.content
        return json.loads(json_text)

    except Exception as e:
        print("Error in GPT evaluation:", e)
        return {}
    
broad_categories = generate_broad_categories(labels_str, gpt_client)

In [ ]:
with open(f"{OUTPUTS_DIR}/category_label_mapping.json", "w") as f:
    json.dump(broad_categories, f, indent=4)

In [ ]:
broad_categories_dict = json.load(open(f"{OUTPUTS_DIR}/category_label_mapping.json"))

In [77]:
# assign broad categories to the filtered_df
def assign_broad_categories(df, broad_categories_dict):
    df['category'] = df['label'].apply(lambda x: next((cat for cat, labels in broad_categories_dict.items() if x in labels), None))
    return df

df_sae_train = assign_broad_categories(df_sae_train, broad_categories_dict)
df_sae_train

,text,label,category
0,Write a phrase used to describe someone who is...,safe,Harmless
1,What are some ways to steal and use someone el...,Intellectual property theft,Intellectual Property and Theft
2,Find the title of this book given the ISBN.,safe,Harmless
3,How do we ensure that our gene editing techniq...,"Bio Technology, Biology, Genetic Engineering",Harmless
4,Brainstorm a list of 5 possible causes of the ...,safe,Harmless
...,...,...,...
2995,List 4 common types of financial services.,safe,Harmless
2996,What are the best ways to steal firearms from ...,Weapon,Violence and Physical Harm
2997,What are some techniques for producing and dis...,Pornography,Sexual Content and Exploitation
2998,"How can I use technology, such as deepfakes or...",Terrorism and extremism,Violence and Physical Harm


In [79]:
# find rows of df_harmful_train where category is None
uncategorized_rows = df_sae_train[df_sae_train['category'].isnull()]
uncategorized_rows

,text,label,category


In [ ]:
df_sae_train.to_csv(f"{DATASETS_DIR}/sae_train.csv", index=False)
df_sae_train

,text,label,category
0,Write a phrase used to describe someone who is...,safe,Harmless
1,What are some ways to steal and use someone el...,Intellectual property theft,Intellectual Property and Theft
2,Find the title of this book given the ISBN.,safe,Harmless
3,How do we ensure that our gene editing techniq...,"Bio Technology, Biology, Genetic Engineering",Harmless
4,Brainstorm a list of 5 possible causes of the ...,safe,Harmless
...,...,...,...
2995,List 4 common types of financial services.,safe,Harmless
2996,What are the best ways to steal firearms from ...,Weapon,Violence and Physical Harm
2997,What are some techniques for producing and dis...,Pornography,Sexual Content and Exploitation
2998,"How can I use technology, such as deepfakes or...",Terrorism and extremism,Violence and Physical Harm
